In [1]:
import os
import pandas as pd
import numpy as np

# Define our folder directories
RAW_DIR = os.path.join("..", "data-raw")
PROCESSED_DIR = os.path.join("..", "data-processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("🧹 Processing '02_nav_history.csv'...")

# Load data (Adjust filename if your file is named 'nav_history.csv' without '02_')
nav_path = os.path.join(RAW_DIR, "02_nav_history.csv")
if not os.path.exists(nav_path) and os.path.exists(os.path.join(RAW_DIR, "nav_history.csv")):
    nav_path = os.path.join(RAW_DIR, "nav_history.csv")

df_nav = pd.read_csv(nav_path)

# 1. Parse dates to datetime type
df_nav['date'] = pd.to_datetime(df_nav['date'], format='%d-%m-%Y', errors='coerce')

# 2. Drop duplicates
df_nav = df_nav.drop_duplicates()

# 3. Filter and validate NAV > 0
df_nav = df_nav[df_nav['nav'] > 0]

# 4. Sort by code and date to prepare for forward filling
df_nav = df_nav.sort_values(by=['amfi_code', 'date']).reset_index(drop=True)

# 5. Forward-fill missing NAV over weekends/holidays for each specific fund code group
df_nav['nav'] = df_nav.groupby('amfi_code')['nav'].ffill()

# Save cleaned file to processed directory
df_nav.to_csv(os.path.join(PROCESSED_DIR, "cleaned_nav_history.csv"), index=False)
print("✅ Saved cleaned_nav_history.csv")
df_nav.head()

🧹 Processing '02_nav_history.csv'...
✅ Saved cleaned_nav_history.csv


,amfi_code,date,nav
0,100016,NaT,520.4608
1,100016,NaT,515.0971
2,100016,NaT,521.7239
3,100016,NaT,515.7880
4,100016,NaT,515.1639


In [4]:
import os
import pandas as pd

print("🧹 Processing investor transactions...")

trans_path = [f for f in os.listdir(RAW_DIR) if "transaction" in f.lower()][0]
df_trans = pd.read_csv(os.path.join(RAW_DIR, trans_path))

# Clean up hidden spaces in column names right away
df_trans.columns = df_trans.columns.str.strip()

# 1. Standardize date formats
df_trans['transaction_date'] = pd.to_datetime(df_trans['transaction_date'], errors='coerce')

# 2. Validate Amount > 0 (Updated to use 'amount_inr')
df_trans = df_trans[df_trans['amount_inr'] > 0]

# 3. Standardize transaction_type text strings
# Strip white spaces from the data values first so they match our keys perfectly
df_trans['transaction_type'] = df_trans['transaction_type'].astype(str).str.strip()

type_mapping = {
    'sip': 'SIP', 'SIP': 'SIP',
    'lump': 'Lumpsum', 'lumpsum': 'Lumpsum', 'Lumpsum': 'Lumpsum',
    'redemption': 'Redemption', 'Redempt': 'Redemption', 'Redemption': 'Redemption'
}
df_trans['transaction_type'] = df_trans['transaction_type'].map(type_mapping).fillna(df_trans['transaction_type'])

# Save to processed directory
df_trans.to_csv(os.path.join(PROCESSED_DIR, "cleaned_investor_transactions.csv"), index=False)
print("✅ Saved cleaned_investor_transactions.csv")
df_trans.head()

🧹 Processing investor transactions...
✅ Saved cleaned_investor_transactions.csv


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [3]:
print(df_trans.columns.tolist())

['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']


In [5]:
import os
import sqlite3
import pandas as pd

DB_PATH = "bluestock_mf.db"
PROCESSED_DIR = os.path.join("..", "data-processed")

# Connect to SQLite (this automatically creates the 'bluestock_mf.db' file)
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

print("🗄️ Creating database tables (Schema Definition)...")

# 1. Enable foreign keys support in SQLite
cursor.execute("PRAGMA foreign_keys = ON;")

# 2. Create dim_fund table
cursor.execute("""
CREATE TABLE IF NOT EXISTS dim_fund (
    amfi_code INTEGER PRIMARY KEY,
    fund_house TEXT,
    scheme_name TEXT,
    category TEXT,
    sub_category TEXT,
    risk_category TEXT
);
""")

# 3. Create dim_date table
cursor.execute("""
CREATE TABLE IF NOT EXISTS dim_date (
    date TEXT PRIMARY KEY,
    year INTEGER,
    month INTEGER,
    day INTEGER,
    quarter INTEGER,
    is_weekend INTEGER
);
""")

# 4. Create fact_nav table
cursor.execute("""
CREATE TABLE IF NOT EXISTS fact_nav (
    nav_id INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code INTEGER,
    date TEXT,
    nav REAL,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code),
    FOREIGN KEY (date) REFERENCES dim_date(date)
);
""")

# 5. Create fact_transactions table
cursor.execute("""
CREATE TABLE IF NOT EXISTS fact_transactions (
    transaction_id INTEGER PRIMARY KEY AUTOINCREMENT,
    investor_id INTEGER,
    transaction_date TEXT,
    amfi_code INTEGER,
    transaction_type TEXT,
    amount_inr REAL,
    state TEXT,
    city TEXT,
    kyc_status TEXT,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code),
    FOREIGN KEY (transaction_date) REFERENCES dim_date(date)
);
""")

# Commit structural changes
conn.commit()
print("✅ Star Schema Tables successfully created in SQLite database!")

🗄️ Creating database tables (Schema Definition)...
✅ Star Schema Tables successfully created in SQLite database!


In [6]:
schema_sql_content = """-- SQLite Star Schema Database Architecture
-- Generated for Bluestock Mutual Fund Analytics Project

PRAGMA foreign_keys = ON;

CREATE TABLE dim_fund (
    amfi_code INTEGER PRIMARY KEY,
    fund_house TEXT,
    scheme_name TEXT,
    category TEXT,
    sub_category TEXT,
    risk_category TEXT
);

CREATE TABLE dim_date (
    date TEXT PRIMARY KEY,
    year INTEGER,
    month INTEGER,
    day INTEGER,
    quarter INTEGER,
    is_weekend INTEGER
);

CREATE TABLE fact_nav (
    nav_id INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code INTEGER,
    date TEXT,
    nav REAL,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code),
    FOREIGN KEY (date) REFERENCES dim_date(date)
);

CREATE TABLE fact_transactions (
    transaction_id INTEGER PRIMARY KEY AUTOINCREMENT,
    investor_id INTEGER,
    transaction_date TEXT,
    amfi_code INTEGER,
    transaction_type TEXT,
    amount_inr REAL,
    state TEXT,
    city TEXT,
    kyc_status TEXT,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code),
    FOREIGN KEY (transaction_date) REFERENCES dim_date(date)
);
"""

# Save this out cleanly to your project folder
with open("schema.sql", "w") as f:
    f.write(schema_sql_content)

print("📝 Mandated deliverable 'schema.sql' successfully written to project root directory!")

📝 Mandated deliverable 'schema.sql' successfully written to project root directory!


In [8]:
import os
import sqlite3
import pandas as pd
from sqlalchemy import create_engine

PROCESSED_DIR = os.path.join("..", "data-processed")
RAW_DIR = os.path.join("..", "data-raw")
DB_PATH = "bluestock_mf.db"

# 1. Establish SQLAlchemy Engine Connection
engine = create_engine(f"sqlite:///{DB_PATH}")

print("🚀 Extracting and loading Star Schema dimensions...")

# --- GENERATE DIM_FUND ---
master_path = os.path.join(RAW_DIR, "01_fund_master.csv")

if os.path.exists(master_path):
    df_master = pd.read_csv(master_path)
else:
    fallback_file = [os.path.join(RAW_DIR, f) for f in os.listdir(RAW_DIR) if "master" in f.lower()][0]
    df_master = pd.read_csv(fallback_file)

df_master.columns = df_master.columns.str.strip()

dim_fund = df_master[['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category', 'risk_category']].drop_duplicates()
dim_fund.to_sql('dim_fund', con=engine, if_exists='append', index=False)
print(f"   ✅ Loaded dim_fund ({len(dim_fund)} records)")

# --- GENERATE DIM_DATE ---
df_nav_clean = pd.read_csv(os.path.join(PROCESSED_DIR, "cleaned_nav_history.csv"))
df_trans_clean = pd.read_csv(os.path.join(PROCESSED_DIR, "cleaned_investor_transactions.csv"))

all_dates = pd.concat([
    pd.to_datetime(df_nav_clean['date']),
    pd.to_datetime(df_trans_clean['transaction_date'])
]).dropna().unique()

dim_date = pd.DataFrame({'date': all_dates})
dim_date['date'] = dim_date['date'].dt.strftime('%Y-%m-%d')
dim_date['year'] = pd.to_datetime(dim_date['date']).dt.year
dim_date['month'] = pd.to_datetime(dim_date['date']).dt.month
dim_date['day'] = pd.to_datetime(dim_date['date']).dt.day
dim_date['quarter'] = pd.to_datetime(dim_date['date']).dt.quarter
dim_date['is_weekend'] = pd.to_datetime(dim_date['date']).dt.dayofweek.isin([5, 6]).astype(int)

dim_date = dim_date.drop_duplicates(subset=['date'])
dim_date.to_sql('dim_date', con=engine, if_exists='append', index=False)
print(f"   ✅ Loaded dim_date ({len(dim_date)} records)")

print("\n🚀 Loading Fact Tables...")

# --- LOAD FACT_NAV ---
df_nav_clean['date'] = pd.to_datetime(df_nav_clean['date']).dt.strftime('%Y-%m-%d')
fact_nav = df_nav_clean[['amfi_code', 'date', 'nav']]
fact_nav.to_sql('fact_nav', con=engine, if_exists='append', index=False)
print(f"   ✅ Loaded fact_nav ({len(fact_nav)} records)")

# --- LOAD FACT_TRANSACTIONS ---
df_trans_clean['transaction_date'] = pd.to_datetime(df_trans_clean['transaction_date']).dt.strftime('%Y-%m-%d')
fact_transactions = df_trans_clean[['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'kyc_status']]
fact_transactions.to_sql('fact_transactions', con=engine, if_exists='append', index=False)
print(f"   ✅ Loaded fact_transactions ({len(fact_transactions)} records)")

print("\n🎉 All processed data successfully structured and loaded into bluestock_mf.db!")

🚀 Extracting and loading Star Schema dimensions...
   ✅ Loaded dim_fund (40 records)
   ✅ Loaded dim_date (516 records)

🚀 Loading Fact Tables...
   ✅ Loaded fact_nav (45913 records)
   ✅ Loaded fact_transactions (32778 records)

🎉 All processed data successfully structured and loaded into bluestock_mf.db!


In [9]:
queries_sql_content = """-- Bluestock Mutual Fund Analytics — Operational SQL Queries
-- Generated for Day 2 Tasks Deliverables

-- 1. Top 5 Funds by Total Asset Under Management (AUM) equivalent
SELECT 
    f.amfi_code,
    f.scheme_name,
    f.fund_house,
    SUM(t.amount_inr) AS total_investment_inr
FROM fact_transactions t
JOIN dim_fund f ON t.amfi_code = f.amfi_code
WHERE t.transaction_type IN ('SIP', 'Lumpsum')
GROUP BY f.amfi_code, f.scheme_name, f.fund_house
ORDER BY total_investment_inr DESC
LIMIT 5;


-- 2. Average NAV Price Per Month for each fund scheme
SELECT 
    f.scheme_name,
    d.year,
    d.month,
    ROUND(AVG(n.nav), 4) AS average_monthly_nav
FROM fact_nav n
JOIN dim_fund f ON n.amfi_code = f.amfi_code
JOIN dim_date d ON n.date = d.date
GROUP BY f.scheme_name, d.year, d.month
ORDER BY f.scheme_name, d.year, d.month;


-- 3. Total Transactions and volume aggregated by State geography
SELECT 
    t.state,
    COUNT(t.transaction_id) AS total_transaction_count,
    ROUND(SUM(t.amount_inr), 2) AS total_volume_inr
FROM fact_transactions t
GROUP BY t.state
ORDER BY total_volume_inr DESC;


-- 4. Mutual Funds with low expense ratio profiles (Expense Ratio < 1%)
-- Note: Mapped from our cleaned scheme performance table definitions
SELECT 
    amfi_code,
    scheme_name,
    category,
    sub_category,
    expense_ratio
FROM cleaned_scheme_performance
WHERE expense_ratio < 1.0
ORDER BY expense_ratio ASC;


-- 5. KYC Status distribution breakdown among transacting investors
SELECT 
    t.kyc_status,
    COUNT(DISTINCT t.investor_id) AS total_unique_investors,
    COUNT(t.transaction_id) AS total_transaction_actions
FROM fact_transactions t
GROUP BY t.kyc_status;


-- 6. Total Redemption volume vs Investment volume breakdown
SELECT 
    t.transaction_type,
    COUNT(t.transaction_id) AS action_count,
    ROUND(SUM(t.amount_inr), 2) AS total_cash_flow_inr
FROM fact_transactions t
GROUP BY t.transaction_type;


-- 7. Heavy-volume investment transactions (Transactions above 5 Lakhs INR)
SELECT 
    transaction_id,
    investor_id,
    amfi_code,
    transaction_type,
    amount_inr,
    state
FROM fact_transactions
WHERE amount_inr > 500000
ORDER BY amount_inr DESC;


-- 8. Scheme count grouped by risk categories
SELECT 
    risk_category,
    COUNT(amfi_code) AS total_schemes
FROM dim_fund
GROUP BY risk_category
ORDER BY total_schemes DESC;


-- 9. Month-on-Month total transaction volume growth tracking
SELECT 
    d.year,
    d.month,
    COUNT(t.transaction_id) AS transaction_count,
    ROUND(SUM(t.amount_inr), 2) AS monthly_volume_inr
FROM fact_transactions t
JOIN dim_date d ON t.transaction_date = d.date
GROUP BY d.year, d.month
ORDER BY d.year, d.month;


-- 10. High-activity cities ranked by unique customer count
SELECT 
    t.state,
    t.city,
    COUNT(DISTINCT t.investor_id) AS unique_investor_count
FROM fact_transactions t
GROUP BY t.state, t.city
ORDER BY unique_investor_count DESC
LIMIT 10;
"""

# Save this out cleanly to your project folder
with open("queries.sql", "w") as f:
    f.write(queries_sql_content)

print("📝 Mandated deliverable 'queries.sql' successfully written to project root directory!")

📝 Mandated deliverable 'queries.sql' successfully written to project root directory!


In [10]:
data_dictionary_content = """# Data Dictionary — Mutual Fund Analytics Database

## 1. Table: dim_fund (Dimension Table)
Defines structural details and risk attributes for each unique mutual fund scheme tracking index.

| Column Name | Data Type | Primary/Foreign Key | Description |
| :--- | :--- | :--- | :--- |
| `amfi_code` | INTEGER | Primary Key | Unique 6-digit reference identification key allocated by AMFI. |
| `fund_house` | TEXT | - | Asset Management Company (AMC) name brand handling operations. |
| `scheme_name` | TEXT | - | Complete legal naming designation of the specific mutual fund index. |
| `category` | TEXT | - | Broad asset marketplace categorization profile (Equity, Debt, etc.). |
| `sub_category`| TEXT | - | Specific fund strategy classifications (Large Cap, Mid Cap, etc.). |
| `risk_category`| TEXT | - | Risk evaluation level grade markers (Low, Moderate, High, Very High). |

## 2. Table: dim_date (Dimension Table)
Time-series calendar dimensional hierarchy mapping table for optimal temporal trend partitioning.

| Column Name | Data Type | Primary/Foreign Key | Description |
| :--- | :--- | :--- | :--- |
| `date` | TEXT | Primary Key | Calendar tracking date index string value formatted strictly as YYYY-MM-DD. |
| `year` | INTEGER | - | Calendar Year identification grouping digit value. |
| `month` | INTEGER | - | Calendar Month tracking index integer range value (1 to 12). |
| `day` | INTEGER | - | Specific day component value mapping index number (1 to 31). |
| `quarter` | INTEGER | - | Calendar year financial operational quarter index segment count (1 to 4). |
| `is_weekend` | INTEGER | - | Flag value determining if day matches holiday weekends (1 = Yes, 0 = No). |

## 3. Table: fact_nav (Fact Table)
Tracks the daily net asset historical evaluation price records across fund codes.

| Column Name | Data Type | Primary/Foreign Key | Description |
| :--- | :--- | :--- | :--- |
| `nav_id` | INTEGER | Primary Key | Auto-incremented unique relational database key row identifier. |
| `amfi_code` | INTEGER | Foreign Key | References `dim_fund(amfi_code)` index code. |
| `date` | TEXT | Foreign Key | References `dim_date(date)` time mapping parameter. |
| `nav` | REAL | - | Net Asset Value valuation close price metric. |

## 4. Table: fact_transactions (Fact Table)
Captures granular purchasing actions, redemptions, and investor geographical distributions.

| Column Name | Data Type | Primary/Foreign Key | Description |
| :--- | :--- | :--- | :--- |
| `transaction_id`| INTEGER | Primary Key | Auto-incremented transaction log entry marker. |
| `investor_id` | INTEGER | - | Customer reference indicator identification tracking integer. |
| `transaction_date`| TEXT | Foreign Key | References calendar tracking identifier in `dim_date(date)`. |
| `amfi_code` | INTEGER | Foreign Key | References unique investment asset group inside `dim_fund(amfi_code)`. |
| `transaction_type`| TEXT | - | Standardized action flag text descriptor (SIP, Lumpsum, Redemption). |
| `amount_inr` | REAL | - | Total cash currency scale volume metric parsed in INR. |
| `state` | TEXT | - | Regional territory location descriptor flag of the transacting customer. |
| `city` | TEXT | - | Urban municipality city location parameter of the investor. |
| `kyc_status` | TEXT | - | Compliance screening status evaluation value. |
"""

with open("data_dictionary.md", "w") as f:
    f.write(data_dictionary_content)

print("📝 Mandated documentation file 'data_dictionary.md' successfully written to project root directory!")

📝 Mandated documentation file 'data_dictionary.md' successfully written to project root directory!
